# Data Preprocessing, EDA & Feature Engineering

mariamzakary


# Cardiovascular Disease Prediction — XAI Project Final
**Dataset:** `cardiovascular_diseases_processed.csv`  
**Papers:** Baghdadi et al. (2023) · Ogunpola et al. (2024) · Tompra et al. (2024)

**Models:** XGBoost · Logistic Regression · Random Forest  
**XAI per model:** SHAP · LIME · PDP · Permutation Importance

---
## 0. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

import shap
!pip install lime
import lime
import lime.lime_tabular

print('All imports OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=f36e2a54ab8f56a8e52e451834cfa52e22a31de24e7ffce506ee2c7e95ccc848
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime
All imports OK


---
## 1. Data Loading, Preprocessing & Feature Engineering

In [2]:
df = pd.read_csv('cardiovascular_diseases_processed.csv')
print(f'Shape: {df.shape}')
df.head(3)

Shape: (68783, 12)


,AGE,GENDER,HEIGHT,WEIGHT,AP_HIGH,AP_LOW,CHOLESTEROL,GLUCOSE,SMOKE,ALCOHOL,PHYSICAL_ACTIVITY,CARDIO_DISEASE
0,50,2,168,62,110,80,1,1,0,0,1,0
1,55,1,156,85,140,90,3,1,0,0,1,1
2,52,1,165,64,130,70,3,1,0,0,0,1


In [3]:
# Outlier capping
df_clean = df.drop(columns=['AGE_GROUP'], errors='ignore').copy()
for col in ['AGE', 'HEIGHT', 'WEIGHT', 'AP_HIGH', 'AP_LOW']:
    lo, hi = df_clean[col].quantile([0.01, 0.99])
    df_clean[col] = df_clean[col].clip(lo, hi)

# Feature engineering
df_clean['BMI']            = df_clean['WEIGHT'] / (df_clean['HEIGHT'] / 100) ** 2
df_clean['PULSE_PRESSURE'] = df_clean['AP_HIGH'] - df_clean['AP_LOW']
df_clean['MAP']            = (df_clean['AP_HIGH'] + 2 * df_clean['AP_LOW']) / 3
df_clean['HYPERTENSION']   = ((df_clean['AP_HIGH'] >= 140) | (df_clean['AP_LOW'] >= 90)).astype(int)
df_clean['OBESE']          = (df_clean['BMI'] >= 30).astype(int)

X = df_clean.drop(columns=['CARDIO_DISEASE'])
y = df_clean['CARDIO_DISEASE']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns)

print(f'Train: {X_train_scaled.shape} | Test: {X_test_scaled.shape}')

Train: (55026, 16) | Test: (13757, 16)
